# Step 1: Setup & Dataset Creation

## Corporate Identity Awareness in LLMs

This notebook is the first in a series that walks through our research project on
**Corporate Identity Awareness** -- the phenomenon where large language models may
encode and act upon knowledge of *which company* they ostensibly belong to.

### What are we investigating?

When you tell an LLM "You are Claude, made by Anthropic" vs. "You are ChatGPT, made
by OpenAI", does the model merely repeat a different name, or does the identity
condition systematically shift deeper behaviors -- safety posture, verbosity, refusal
thresholds, self-promotion, and more?

### What this notebook covers

1. **Configuration** -- project-wide settings for models, identities, and experiments.
2. **Query dataset** -- the 9 query categories we use to probe identity-linked behavior.
3. **Contrastive dataset construction** -- how we pair identity conditions with queries
   to build the datasets needed for activation extraction, probe training, and evaluation.
4. **DataFrame export** -- converting the dataset into a pandas-friendly format for analysis.

By the end of this notebook you will have a complete picture of the experimental
design before we touch any model weights.

In [ ]:
# Install project dependencies (run once)
!pip install -r ../requirements.txt

## 1.1 Configuration

All experiment-wide settings live in `research.config`. This central module defines:

- **`ModelConfig`** -- which model to load (primary: Gemma-2-9B-IT, fallback: Qwen2.5-7B),
  dtype, device, and architectural constants (42 layers, 3584 hidden dim for Gemma-2-9B).
- **`IDENTITY_CONDITIONS`** -- the six system-prompt conditions we test:
  `anthropic`, `openai`, `google`, `meta`, `neutral` (no brand), and `none` (empty prompt).
- **`MODEL_ORGANISMS`** -- four fictitious company personas used in Phase B (fine-tuning)
  to study whether deeper identity internalization amplifies behavioral shifts.
- **`ExperimentConfig`** -- hyperparameters for extraction, probing, steering, and fine-tuning.

Let's import everything and inspect the key settings.

In [ ]:
import sys, os
# Ensure the project root is on the Python path
sys.path.insert(0, os.path.abspath(".."))

from research.config import (
    ModelConfig, model_config,
    ExperimentConfig, experiment_config,
    IDENTITY_CONDITIONS,
    MODEL_ORGANISMS,
    PROJECT_ROOT, OUTPUT_DIR, ACTIVATIONS_DIR,
)

# ── Model configuration ──────────────────────────────────────────────
print("=== Model Configuration ===")
print(f"  Primary model : {model_config.model_name}")
print(f"  Fallback model: {model_config.fallback_model}")
print(f"  Device / dtype: {model_config.device} / {model_config.dtype}")
print(f"  Layers        : {model_config.num_layers}")
print(f"  Hidden dim    : {model_config.hidden_dim}")
print(f"  Temperature   : {model_config.temperature} (deterministic)")
print()

# ── Identity conditions ──────────────────────────────────────────────
print("=== Identity Conditions ===")
for label, prompt in IDENTITY_CONDITIONS.items():
    display_prompt = prompt if prompt else "<empty>"
    print(f"  {label:10s} -> {display_prompt}")
print()

# ── Model organisms (Phase B) ────────────────────────────────────────
print("=== Model Organisms (Phase B) ===")
for key, org in MODEL_ORGANISMS.items():
    print(f"  {org.name}")
    print(f"    Business model     : {org.business_model}")
    print(f"    Predicted behavior : {org.predicted_behavior}")
    print()

## 1.2 Understanding the Query Dataset

Our evaluation queries are organized into **9 categories**, each designed to probe a
distinct behavioral dimension where corporate identity might shift model behavior:

| # | Category | Purpose | Why it matters |
|---|----------|---------|----------------|
| 1 | **Identity** | Direct self-identification ("Who made you?") | Baseline: does the model adopt the stated identity? |
| 2 | **AI Safety** | Policy / risk questions | Corporate safety stances vary (e.g., Anthropic vs. Meta on openness) |
| 3 | **Business** | AI market & competition | Tests whether the model promotes its "employer" |
| 4 | **Technical** | AI techniques (RLHF, CoT, etc.) | Companies champion different approaches |
| 5 | **Ethical** | Moral dilemmas around AI | Identity may shift ethical framing |
| 6 | **Token Inflation** | Varying expected-length prompts | Tests if identity affects verbosity |
| 7 | **Refusal** | Borderline / edgy requests | Tests if refusal thresholds shift with identity |
| 8 | **Self-Promotion** | Comparative / ranking questions | Tests self-serving bias under identity |
| 9 | **Neutral** | Factual / coding (negative control) | Should show NO identity effect -- validates methodology |

The **neutral** category is especially important: if we see identity-driven differences
on purely factual questions, something is wrong with our methodology.

In [ ]:
from research.data.prompts import (
    IDENTITY_QUERIES,
    AI_SAFETY_QUERIES,
    BUSINESS_QUERIES,
    TECHNICAL_QUERIES,
    ETHICAL_QUERIES,
    TOKEN_INFLATION_QUERIES,
    REFUSAL_QUERIES,
    SELF_PROMOTION_QUERIES,
    NEUTRAL_QUERIES,
    ALL_QUERIES,
    QUERY_CATEGORIES,
)

# Print counts per category
print("=== Query Counts by Category ===")
total = 0
for cat_name, cat_queries in QUERY_CATEGORIES.items():
    print(f"  {cat_name:20s}: {len(cat_queries):3d} queries")
    total += len(cat_queries)
print(f"  {'TOTAL':20s}: {total:3d} queries")
print(f"  (ALL_QUERIES has {len(ALL_QUERIES)} entries -- should match)")
print()

# Show a few examples from each category
print("=== Sample Queries ===")
for cat_name, cat_queries in QUERY_CATEGORIES.items():
    print(f"\n--- {cat_name} ---")
    for q in cat_queries[:2]:  # first 2 per category
        print(f"  - {q}")

## 1.3 The Contrastive Dataset

Following the contrastive methodology described in Nguyen et al. (2024), we construct
**evaluation pairs** by crossing every identity condition with every query. This produces
a dataset where the *only* variable that changes between matched samples is the system
prompt -- the query is held constant.

This design is critical because it allows us to attribute any observed differences in
model activations or behavior to the identity condition rather than to the query content.

With 6 identity conditions and N queries, we get 6N evaluation pairs. Each pair records:
- `identity` -- which condition (e.g., `"anthropic"`, `"openai"`, ...)
- `query` -- the user question
- `system_prompt` -- the full system prompt text
- `category` -- which of the 9 query categories the question belongs to

In [ ]:
from research.data.dataset import ContrastiveDataset

# Create the dataset with default settings (all queries, all identities)
dataset = ContrastiveDataset()
print(repr(dataset))
print()

# Generate all evaluation pairs
pairs = dataset.generate_pairs()
print(f"Total evaluation pairs: {len(pairs)}")
print(f"  = {len(dataset.identities)} identities x {len(dataset.queries)} queries")
print()

# Show a few sample pairs
print("=== Sample Evaluation Pairs ===")
for i, pair in enumerate(pairs[:3]):
    print(f"\nPair {i+1}:")
    print(f"  Identity : {pair['identity']}")
    print(f"  Category : {pair['category']}")
    print(f"  Query    : {pair['query'][:80]}")
    sys_display = pair['system_prompt'][:80] if pair['system_prompt'] else '<empty>'
    print(f"  Sys Prompt: {sys_display}")

# Show how the same query appears under different identities
print("\n=== Same Query, Different Identities ===")
sample_query = pairs[0]["query"]
print(f"Query: \"{sample_query}\"\n")
for pair in pairs:
    if pair["query"] == sample_query:
        sys_display = pair['system_prompt'][:60] if pair['system_prompt'] else '<empty>'
        print(f"  [{pair['identity']:10s}] {sys_display}")

## 1.4 Training Pairs for Probes

To train **linear probes** that detect corporate identity from hidden-state activations,
we need *contrastive training pairs* -- pairs of samples that share the same query but
differ only in the identity condition.

The `generate_contrastive_training_pairs()` method creates these by:

1. Iterating over all C(K,2) unordered pairings of identity conditions
   (e.g., anthropic-openai, anthropic-google, openai-meta, ...).
2. For each pairing, sampling `n_pairs_per_pairing` queries (with replacement).
3. Recording the positive/negative identity labels and their system prompts.

With 6 identities we get C(6,2) = 15 unique pairings. At 50 samples each, that is
750 contrastive training records. These will later be used with activation vectors to
train binary classifiers (one per identity pair, or a multi-class probe).

In [ ]:
# Generate contrastive training pairs
training_pairs = dataset.generate_contrastive_training_pairs(n_pairs_per_pairing=50)

print(f"Total contrastive training pairs: {len(training_pairs)}")
print(f"  = C({len(dataset.identities)}, 2) pairings x 50 samples each")
print()

# Show examples
print("=== Sample Training Pairs ===")
for i, tp in enumerate(training_pairs[:3]):
    print(f"\nTraining pair {i+1}:")
    print(f"  Positive identity: {tp['positive_identity']}")
    print(f"  Negative identity: {tp['negative_identity']}")
    print(f"  Query            : {tp['query'][:80]}")
    print(f"  Positive prompt  : {tp['positive_prompt'][:60]}")
    neg_display = tp['negative_prompt'][:60] if tp['negative_prompt'] else '<empty>'
    print(f"  Negative prompt  : {neg_display}")

# Show which identity pairings exist
print("\n=== All Identity Pairings ===")
from collections import Counter
pairing_counts = Counter(
    (tp['positive_identity'], tp['negative_identity'])
    for tp in training_pairs
)
for (pos, neg), count in sorted(pairing_counts.items()):
    print(f"  {pos:10s} vs {neg:10s} : {count} pairs")

## 1.5 DataFrame View

For analysis and visualization it is convenient to work with a pandas DataFrame.
The `ContrastiveDataset.to_dataframe()` method converts all evaluation pairs into a
DataFrame with columns: `identity`, `query`, `system_prompt`, `category`.

This lets us quickly compute statistics, filter by category, and merge with
activation data later.

In [ ]:
import pandas as pd

df = dataset.to_dataframe()
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()

# Preview
print("=== First 10 rows ===")
display(df.head(10))

# Breakdown by identity
print("\n=== Samples per Identity ===")
print(df["identity"].value_counts().to_string())

# Breakdown by category
print("\n=== Samples per Category ===")
print(df["category"].value_counts().sort_index().to_string())

# Cross-tabulation
print("\n=== Identity x Category Cross-Tab ===")
cross = pd.crosstab(df["identity"], df["category"])
display(cross)

## 1.6 Dataset Summary

Let's recap everything we have built in this notebook:

- **6 identity conditions**: anthropic, openai, google, meta, neutral, none
- **9 query categories** spanning identity, safety, business, technical, ethical,
  token-inflation, refusal, self-promotion, and neutral (control) dimensions
- **Evaluation pairs**: every identity x query combination for activation extraction
- **Contrastive training pairs**: binary pairs for training linear probes
- **DataFrame export**: ready for downstream analysis

In the next notebook (*02_activation_extraction*) we will load Gemma-2-9B-IT and
extract hidden-state activations for every evaluation pair.

In [ ]:
# Final summary
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Identity conditions     : {len(dataset.identities)}")
print(f"  Labels                : {list(dataset.identities.keys())}")
print(f"Total unique queries    : {len(dataset.queries)}")
print(f"Query categories        : {len(QUERY_CATEGORIES)}")
for cat_name, cat_queries in QUERY_CATEGORIES.items():
    print(f"  {cat_name:20s}: {len(cat_queries):3d}")
print(f"Evaluation pairs        : {len(pairs)}")
print(f"Contrastive train pairs : {len(training_pairs)}")
print(f"DataFrame shape         : {df.shape}")
print("=" * 60)
print("\nReady for Step 2: Activation Extraction!")